# PharmaDoc Intelligence: End-to-End Pipeline Evaluation

This notebook implements and evaluates an AI-powered pipeline for processing digital and scanned pharmaceutical PDFs. It combines OCR, document classification, metadata-aware chunking, FAISS retrieval, an open-source language model, and a Gradio interface to produce source-grounded answers.

## 1. Setup and Installation

In [ ]:
# Install the Tesseract OCR engine for scanned PDFs
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

# Maintain compatibility with Google Colab
!pip install -q \
    "numpy>=1.24,<2.3" \
    "pandas==2.2.3" \
    "setuptools<82"

# Install document-processing and RAG libraries
!pip install -q \
    gradio \
    gradio_pdf \
    pypdf \
    pymupdf \
    pytesseract \
    pillow \
    faiss-cpu \
    sentence-transformers \
    "transformers>=4.43,<5" \
    "accelerate>=0.31,<2" \
    sentencepiece \
    llama-index-core

print("Installation complete.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━

## 2. Imports and Configuration

In [ ]:
import io
import json
import hashlib
import re
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import gradio as gr
from gradio_pdf import PDF

import pymupdf
from pypdf import PdfReader
import pytesseract
from PIL import Image

import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter


MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
DEFAULT_TOP_K = 4
OCR_TEXT_THRESHOLD = 50

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Select Runtime → Change runtime type → T4 GPU."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False
)

language_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=False,
    low_cpu_mem_usage=True,
    attn_implementation="eager"
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

language_model.eval()

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device="cpu"
)

print("Models loaded successfully.")

GPU: Tesla T4
Loading microsoft/Phi-3.5-mini-instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded successfully.


## 3. Data Structures

In [ ]:
DOCUMENT_TYPES = [
    "Research Article",
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other"
]


@dataclass
class PageRecord:
    source_id: str
    file_name: str
    page_number: int
    text: str
    extraction_method: str


@dataclass
class LogicalDocument:
    document_id: str
    source_id: str
    file_name: str
    document_type: str
    page_start: int
    page_end: int
    text: str


@dataclass
class DocumentChunk:
    chunk_id: str
    document_id: str
    source_id: str
    file_name: str
    document_type: str
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = field(
        default=None,
        repr=False
    )

    def metadata(self) -> Dict[str, Any]:
        return {
            "chunk_id": self.chunk_id,
            "document_id": self.document_id,
            "source_id": self.source_id,
            "file_name": self.file_name,
            "document_type": self.document_type,
            "page_start": self.page_start,
            "page_end": self.page_end
        }


@dataclass
class FileSummary:
    source_id: str
    file_name: str
    total_pages: int
    digital_pages: int
    ocr_pages: int
    document_count: int = 0
    chunk_count: int = 0
    processing_seconds: float = 0.0


print("Data structures ready.")

Data structures ready.


## 4. PDF Extraction and OCR

In [ ]:
def clean_extracted_text(text: str) -> str:
    if not text:
        return ""

    text = text.replace("\x00", " ")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    cleaned_lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def resolve_file_path(file_input: Any) -> str:
    if isinstance(file_input, (str, Path)):
        return str(file_input)

    if isinstance(file_input, dict):
        path = file_input.get("path") or file_input.get("name")
        if path:
            return str(path)

    if hasattr(file_input, "name"):
        return str(file_input.name)

    raise ValueError("The uploaded file path could not be resolved.")


def run_ocr(page: pymupdf.Page) -> str:
    zoom = 2.5
    matrix = pymupdf.Matrix(zoom, zoom)

    pixmap = page.get_pixmap(
        matrix=matrix,
        alpha=False
    )

    image = Image.open(
        io.BytesIO(pixmap.tobytes("png"))
    ).convert("RGB")

    # Detect and correct page orientation before OCR.
    try:
        orientation = pytesseract.image_to_osd(
            image,
            output_type=pytesseract.Output.DICT
        )

        rotation = int(
            orientation.get("rotate", 0)
        )

        if rotation:
            image = image.rotate(
                -rotation,
                expand=True
            )

    except Exception:
        # Orientation detection may fail when there is too little
        # readable text. In that case, keep the original orientation.
        pass

    text = pytesseract.image_to_string(
        image,
        lang="eng",
        config="--psm 3"
    )

    return clean_extracted_text(text)


def extract_pdf_pages(
    file_input: Any
) -> Tuple[List[PageRecord], FileSummary]:

    start_time = time.perf_counter()
    file_path = resolve_file_path(file_input)
    file_name = Path(file_path).name

    with open(file_path, "rb") as file:
        pdf_bytes = file.read()

    source_id = hashlib.sha256(pdf_bytes).hexdigest()[:12]

    try:
        pdf_reader = PdfReader(io.BytesIO(pdf_bytes))
        if pdf_reader.is_encrypted:
            raise ValueError(
                f"{file_name} is password protected and cannot be processed."
            )
    except ValueError:
        raise
    except Exception as error:
        raise ValueError(
            f"{file_name} could not be read as a valid PDF."
        ) from error

    pages = []
    digital_pages = 0
    ocr_pages = 0

    with pymupdf.open(
        stream=pdf_bytes,
        filetype="pdf"
    ) as pdf_document:

        for page_index, page in enumerate(pdf_document):
            digital_text = clean_extracted_text(
                page.get_text("text")
            )

            if len(digital_text) >= OCR_TEXT_THRESHOLD:
                final_text = digital_text
                extraction_method = "Digital"
                digital_pages += 1
            else:
                ocr_text = run_ocr(page)

                if len(ocr_text) > len(digital_text):
                    final_text = ocr_text
                    extraction_method = "OCR"
                    ocr_pages += 1
                else:
                    final_text = digital_text
                    extraction_method = "Digital"
                    digital_pages += 1

            pages.append(
                PageRecord(
                    source_id=source_id,
                    file_name=file_name,
                    page_number=page_index + 1,
                    text=final_text,
                    extraction_method=extraction_method
                )
            )

    summary = FileSummary(
        source_id=source_id,
        file_name=file_name,
        total_pages=len(pages),
        digital_pages=digital_pages,
        ocr_pages=ocr_pages,
        processing_seconds=round(
            time.perf_counter() - start_time,
            2
        )
    )

    return pages, summary


def extract_multiple_pdfs(
    uploaded_files: List[Any]
) -> Tuple[List[PageRecord], List[FileSummary]]:

    if not uploaded_files:
        raise ValueError("Upload at least one PDF.")

    all_pages = []
    summaries = []

    for uploaded_file in uploaded_files:
        pages, summary = extract_pdf_pages(uploaded_file)
        all_pages.extend(pages)
        summaries.append(summary)

    return all_pages, summaries


print("PDF extraction and OCR functions ready.")

PDF extraction and OCR functions ready.


## 5. Document Classification and Segmentation

In [ ]:
DOCUMENT_HEADINGS = {
    "Cover Letter": [
        "cover letter",
        "dear sir",
        "dear madam",
        "to whom it may concern",
        "please find enclosed"
    ],
    "Certificate of Quality": [
        "certificate of quality",
        "certificate of analysis",
        "certificate of conformance",
        "certificate of conformity",
        "quality certificate"
    ],
    "Packaging Specification": [
        "packaging specification",
        "packaging component specification",
        "container closure specification",
        "label specification"
    ],
    "BSE/TSE Declaration": [
        "bse/tse declaration",
        "bse tse declaration",
        "tse declaration",
        "transmissible spongiform encephalopathy declaration"
    ],
    "Material Description": [
        "material description",
        "material specification",
        "materials of construction",
        "material composition"
    ],
    "Supplier Qualification": [
        "supplier qualification",
        "supplier approval",
        "approved supplier",
        "vendor qualification"
    ],
    "Chain of Custody": [
        "global chain of custody",
        "chain of custody",
        "custody record",
        "traceability record"
    ]
}


DOCUMENT_KEYWORDS = {
    "Cover Letter": [
        "dear sir",
        "dear madam",
        "to whom it may concern",
        "please find enclosed",
        "enclosed documents",
        "sincerely"
    ],
    "Certificate of Quality": [
        "lot number",
        "batch number",
        "article number",
        "date of manufacture",
        "manufacture date",
        "expiry date",
        "expiration date",
        "release criteria",
        "test results",
        "meets the specified criteria",
        "critical quality attributes"
    ],
    "Packaging Specification": [
        "packaging component",
        "container closure",
        "pack size",
        "label specification",
        "carton specification",
        "package configuration"
    ],
    "BSE/TSE Declaration": [
        "bse/tse",
        "bse tse",
        "transmissible spongiform",
        "bovine spongiform",
        "animal-derived materials",
        "animal origin"
    ],
    "Material Description": [
        "materials of construction",
        "chemical composition",
        "polymeric materials",
        "physical properties",
        "sterilisation compatibility",
        "sterilization compatibility"
    ],
    "Supplier Qualification": [
        "supplier qualification",
        "supplier audit",
        "approved supplier",
        "supplier status",
        "approval status",
        "approved until",
        "vendor qualification"
    ],
    "Chain of Custody": [
        "chain of custody",
        "shipment traceability",
        "material traceability",
        "transfer record",
        "received by",
        "released by",
        "raw materials to distribution"
    ]
}


def generate_text(
    system_prompt: str,
    user_prompt: str,
    max_new_tokens: int = 200
) -> str:
    """Generate text with the open-source Phi-3.5 model."""

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(language_model.device)

    with torch.inference_mode():
        output_ids = language_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = output_ids[
        0,
        model_inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()


def normalise_for_classification(
    text: str
) -> str:
    """Normalise extracted text without removing useful wording."""

    return re.sub(
        r"\s+",
        " ",
        clean_extracted_text(text).lower()
    ).strip()


def looks_like_research_article(
    text: str
) -> bool:
    """Identify structural features commonly found in research papers."""

    normalised_text = normalise_for_classification(
        text
    )

    research_markers = [
        "abstract",
        "introduction",
        "methodology",
        "methods",
        "results",
        "discussion",
        "conclusion",
        "references",
        "bibliography",
        "doi",
        "journal",
        "et al."
    ]

    marker_count = sum(
        marker in normalised_text
        for marker in research_markers
    )

    return marker_count >= 4


def detect_heading_type(
    text: str
) -> Optional[str]:
    """Find the earliest strong document heading on a page."""

    searchable_text = normalise_for_classification(
        text
    )[:3000]

    matches = []

    for document_type, headings in (
        DOCUMENT_HEADINGS.items()
    ):
        for heading in headings:
            position = searchable_text.find(
                heading
            )

            if position >= 0:
                matches.append(
                    (
                        position,
                        document_type
                    )
                )

    if not matches:
        return None

    matches.sort(
        key=lambda item: item[0]
    )

    return matches[0][1]


def score_document_types(
    text: str
) -> Dict[str, int]:
    """Score supporting evidence for every pharmaceutical type."""

    text_lower = normalise_for_classification(
        text
    )

    beginning = text_lower[:1800]
    scores = {}

    for document_type, keywords in (
        DOCUMENT_KEYWORDS.items()
    ):
        score = 0

        for keyword in keywords:
            if keyword in beginning:
                score += 2
            elif keyword in text_lower:
                score += 1

        scores[document_type] = score

    return scores


def classify_page_type(
    text: str
) -> Tuple[str, int, bool]:
    """Return the page type, evidence score, and heading status."""

    heading_type = detect_heading_type(
        text
    )

    if heading_type is not None:
        return (
            heading_type,
            10,
            True
        )

    scores = score_document_types(
        text
    )

    best_type = max(
        scores,
        key=scores.get
    )

    best_score = scores[
        best_type
    ]

    if best_score < 2:
        return (
            "Other",
            best_score,
            False
        )

    return (
        best_type,
        best_score,
        False
    )


def keyword_document_type(
    text: str
) -> str:
    """Classify pharmaceutical text using headings and evidence."""

    document_type, _, _ = classify_page_type(
        text
    )

    return document_type


def classify_document_type(
    text: str
) -> str:
    """Return one consistent document-type string."""

    if looks_like_research_article(
        text
    ):
        return "Research Article"

    return keyword_document_type(
        text
    )


def build_logical_document(
    segment: List[PageRecord],
    document_type_hint: Optional[str] = None
) -> LogicalDocument:
    """Build one logical document with consistent metadata."""

    combined_text = "\n\n".join(
        f"[Page {page.page_number}]\n"
        f"{page.text}"
        for page in segment
        if page.text
    )

    document_type = (
        document_type_hint
        if document_type_hint
        and document_type_hint != "Other"
        else classify_document_type(
            combined_text
        )
    )

    identifier = (
        f"{segment[0].source_id}:"
        f"{segment[0].page_number}:"
        f"{segment[-1].page_number}:"
        f"{document_type}"
    )

    document_id = hashlib.sha256(
        identifier.encode("utf-8")
    ).hexdigest()[:16]

    return LogicalDocument(
        document_id=document_id,
        source_id=segment[0].source_id,
        file_name=segment[0].file_name,
        document_type=document_type,
        page_start=segment[0].page_number,
        page_end=segment[-1].page_number,
        text=combined_text
    )


def segment_file_pages(
    pages: List[PageRecord]
) -> List[LogicalDocument]:
    """Segment one PDF using page headings and page-level evidence."""

    if not pages:
        return []

    ordered_pages = sorted(
        pages,
        key=lambda page: page.page_number
    )

    whole_file_text = "\n\n".join(
        page.text
        for page in ordered_pages
        if page.text
    )

    # Research papers normally form one continuous document.
    if looks_like_research_article(
        whole_file_text
    ):
        return [
            build_logical_document(
                ordered_pages,
                "Research Article"
            )
        ]

    segments = []
    current_segment = []
    current_type = "Other"

    for page in ordered_pages:
        (
            page_type,
            page_score,
            has_heading
        ) = classify_page_type(
            page.text
        )

        starts_new_document = (
            bool(current_segment)
            and (
                has_heading
                or (
                    page_type != "Other"
                    and page_type != current_type
                    and page_score >= 4
                )
            )
        )

        if starts_new_document:
            segments.append(
                (
                    current_segment,
                    current_type
                )
            )

            current_segment = []
            current_type = "Other"

        current_segment.append(
            page
        )

        if page_type != "Other":
            if (
                current_type == "Other"
                or has_heading
            ):
                current_type = page_type

    if current_segment:
        segments.append(
            (
                current_segment,
                current_type
            )
        )

    return [
        build_logical_document(
            segment,
            document_type_hint
        )
        for segment, document_type_hint
        in segments
    ]


def segment_all_documents(
    pages: List[PageRecord],
    summaries: Optional[
        List[FileSummary]
    ] = None
) -> List[LogicalDocument]:
    """Segment every uploaded file and update its summary."""

    grouped_pages = {}

    for page in pages:
        grouped_pages.setdefault(
            page.source_id,
            []
        ).append(
            page
        )

    logical_documents = []

    for source_pages in grouped_pages.values():
        logical_documents.extend(
            segment_file_pages(
                source_pages
            )
        )

    if summaries is not None:
        for summary in summaries:
            summary.document_count = sum(
                document.source_id
                == summary.source_id
                for document
                in logical_documents
            )

    return logical_documents


print(
    "Document classification and segmentation functions ready."
)

Document classification and segmentation functions ready.


 ## 6. Chunking and Metadata

In [ ]:
sentence_splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    paragraph_separator="\n\n",
    separator=" "
)


def extract_page_sections(
    logical_document: LogicalDocument
) -> List[Tuple[int, str]]:

    page_pattern = re.compile(r"\[Page (\d+)\]\n")
    matches = list(
        page_pattern.finditer(logical_document.text)
    )

    if not matches:
        return [
            (
                logical_document.page_start,
                logical_document.text
            )
        ]

    page_sections = []

    for index, match in enumerate(matches):
        page_number = int(match.group(1))
        text_start = match.end()

        if index + 1 < len(matches):
            text_end = matches[index + 1].start()
        else:
            text_end = len(logical_document.text)

        page_text = logical_document.text[
            text_start:text_end
        ].strip()

        if page_text:
            page_sections.append(
                (page_number, page_text)
            )

    return page_sections


def chunk_logical_document(
    logical_document: LogicalDocument
) -> List[DocumentChunk]:

    chunks = []
    chunk_index = 0

    for page_number, page_text in extract_page_sections(
        logical_document
    ):
        text_chunks = sentence_splitter.split_text(
            page_text
        )

        for chunk_text in text_chunks:
            chunk_text = chunk_text.strip()

            if not chunk_text:
                continue

            identifier = (
                f"{logical_document.document_id}:"
                f"{page_number}:"
                f"{chunk_index}"
            )

            chunk_id = hashlib.sha256(
                identifier.encode("utf-8")
            ).hexdigest()[:20]

            chunks.append(
                DocumentChunk(
                    chunk_id=chunk_id,
                    document_id=logical_document.document_id,
                    source_id=logical_document.source_id,
                    file_name=logical_document.file_name,
                    document_type=logical_document.document_type,
                    page_start=page_number,
                    page_end=page_number,
                    text=chunk_text
                )
            )

            chunk_index += 1

    return chunks


def create_document_chunks(
    logical_documents: List[LogicalDocument],
    summaries: Optional[List[FileSummary]] = None
) -> List[DocumentChunk]:

    all_chunks = []

    for logical_document in logical_documents:
        all_chunks.extend(
            chunk_logical_document(logical_document)
        )

    if summaries is not None:
        for summary in summaries:
            summary.chunk_count = sum(
                chunk.source_id == summary.source_id
                for chunk in all_chunks
            )

    return all_chunks


print("Chunking and metadata functions ready.")

Chunking and metadata functions ready.


## 7. Embeddings and FAISS Indexing

In [ ]:
class RAGIndex:
    def __init__(self):
        self.index = None
        self.chunks = []
        self.embeddings = None
        self.document_type_indices = {}

    @property
    def is_ready(self) -> bool:
        return self.index is not None and bool(self.chunks)

    def clear(self):
        self.index = None
        self.chunks = []
        self.embeddings = None
        self.document_type_indices = {}

    def build(self, chunks: List[DocumentChunk]):
        if not chunks:
            raise ValueError(
                "No document chunks were available for indexing."
            )

        self.clear()
        self.chunks = chunks

        texts = [chunk.text for chunk in chunks]

        embeddings = embedding_model.encode(
            texts,
            batch_size=32,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        self.embeddings = np.ascontiguousarray(
            embeddings
        )

        dimension = self.embeddings.shape[1]

        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(self.embeddings)

        for index, chunk in enumerate(self.chunks):
            chunk.embedding = self.embeddings[index]

        document_types = sorted(
            set(chunk.document_type for chunk in chunks)
        )

        for document_type in document_types:
            positions = [
                index
                for index, chunk in enumerate(chunks)
                if chunk.document_type == document_type
            ]

            type_embeddings = np.ascontiguousarray(
                self.embeddings[positions]
            )

            type_index = faiss.IndexFlatIP(dimension)
            type_index.add(type_embeddings)

            self.document_type_indices[document_type] = {
                "index": type_index,
                "positions": positions
            }

        print(
            f"Indexed {len(chunks)} chunks across "
            f"{len(document_types)} document types."
        )

    def search(
        self,
        query: str,
        top_k: int = DEFAULT_TOP_K,
        document_type: Optional[str] = None
    ) -> List[Tuple[DocumentChunk, float]]:

        if not self.is_ready:
            raise ValueError(
                "Process and index at least one PDF before searching."
            )

        query_embedding = embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        ).astype("float32")

        query_embedding = np.ascontiguousarray(
            query_embedding
        )

        if (
            document_type
            and document_type != "All Documents"
            and document_type in self.document_type_indices
        ):
            type_data = self.document_type_indices[
                document_type
            ]

            search_index = type_data["index"]
            positions = type_data["positions"]
        else:
            search_index = self.index
            positions = list(range(len(self.chunks)))

        result_count = min(
            int(top_k),
            search_index.ntotal
        )

        if result_count == 0:
            return []

        scores, indices = search_index.search(
            query_embedding,
            result_count
        )

        results = []

        for score, local_index in zip(
            scores[0],
            indices[0]
        ):
            if local_index < 0:
                continue

            original_index = positions[local_index]
            chunk = self.chunks[original_index]

            similarity = max(
                0.0,
                min(1.0, float(score))
            )

            results.append(
                (chunk, similarity)
            )

        return results


rag_index = RAGIndex()

print("FAISS index ready.")

FAISS index ready.


## 8. Query Routing and Retrieval

In [ ]:
ROUTING_DESCRIPTIONS = {
    "Research Article": (
        "Questions about academic and scientific research papers, "
        "including abstracts, authors, methods, results, findings, "
        "discussions, conclusions, publication details, journals, "
        "DOIs, and references."
    ),
    "Cover Letter": (
        "Questions about senders, recipients, enclosed documents, "
        "correspondence, product information, or the purpose of a letter."
    ),
    "Certificate of Quality": (
        "Questions about lot numbers, batch numbers, manufacture dates, "
        "expiry dates, specifications, test results, assay, or purity."
    ),
    "Packaging Specification": (
        "Questions about packaging components, containers, closures, "
        "labels, cartons, pack sizes, or packaging part numbers."
    ),
    "BSE/TSE Declaration": (
        "Questions about animal-derived materials, bovine origin, "
        "BSE, TSE, or transmissible spongiform encephalopathy compliance."
    ),
    "Material Description": (
        "Questions about material composition, construction, polymers, "
        "physical properties, or sterilisation compatibility."
    ),
    "Supplier Qualification": (
        "Questions about suppliers, vendors, audits, approvals, "
        "quality systems, or ISO certification."
    ),
    "Chain of Custody": (
        "Questions about traceability, transfers, shipments, custody, "
        "receipt, release, or movement of materials."
    )
}


QUERY_ROUTING_KEYWORDS = {
    "Research Article": [
        "research",
        "research article",
        "paper",
        "study",
        "abstract",
        "author",
        "authors",
        "method",
        "methods",
        "methodology",
        "result",
        "results",
        "finding",
        "findings",
        "discussion",
        "conclusion",
        "published",
        "publication",
        "publication date",
        "journal",
        "doi"
    ],
    "Cover Letter": [
        "cover letter",
        "sender",
        "recipient",
        "enclosed",
        "correspondence"
    ],
    "Certificate of Quality": [
        "lot number",
        "batch number",
        "manufacture date",
        "expiry date",
        "expiration date",
        "test result",
        "assay",
        "purity"
    ],
    "Packaging Specification": [
        "packaging",
        "pack size",
        "container",
        "closure",
        "label",
        "carton",
        "part number"
    ],
    "BSE/TSE Declaration": [
        "bse",
        "tse",
        "animal origin",
        "bovine",
        "spongiform"
    ],
    "Material Description": [
        "material composition",
        "material description",
        "construction",
        "polymer",
        "sterilisation"
    ],
    "Supplier Qualification": [
        "supplier",
        "vendor",
        "audit",
        "approved product",
        "iso certification"
    ],
    "Chain of Custody": [
        "chain of custody",
        "traceability",
        "shipment",
        "transfer",
        "received by",
        "released by"
    ]
}


@dataclass
class RoutingDecision:
    document_type: str
    confidence: float
    method: str
    filter_applied: bool


routing_types = list(
    ROUTING_DESCRIPTIONS.keys()
)

routing_embeddings = embedding_model.encode(
    list(ROUTING_DESCRIPTIONS.values()),
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")


def normalise_routing_text(
    text: str
) -> str:

    normalised_text = re.sub(
        r"[-_/]+",
        " ",
        text.lower()
    )

    return re.sub(
        r"\s+",
        " ",
        normalised_text
    ).strip()


def route_query(
    query: str
) -> RoutingDecision:

    query_lower = normalise_routing_text(
        query
    )

    keyword_scores = {}

    for document_type, keywords in (
        QUERY_ROUTING_KEYWORDS.items()
    ):
        keyword_scores[document_type] = sum(
            normalise_routing_text(keyword)
            in query_lower
            for keyword in keywords
        )

    best_keyword_type = max(
        keyword_scores,
        key=keyword_scores.get
    )

    best_keyword_score = keyword_scores[
        best_keyword_type
    ]

    if best_keyword_score > 0:
        confidence = min(
            0.95,
            0.72 + (
                best_keyword_score * 0.08
            )
        )

        return RoutingDecision(
            document_type=best_keyword_type,
            confidence=confidence,
            method="Keyword",
            filter_applied=False
        )

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")[0]

    similarities = (
        routing_embeddings @ query_embedding
    )

    ranked_indices = np.argsort(
        similarities
    )[::-1]

    best_index = int(
        ranked_indices[0]
    )

    second_index = int(
        ranked_indices[1]
    )

    best_score = float(
        similarities[best_index]
    )

    second_score = float(
        similarities[second_index]
    )

    score_margin = (
        best_score - second_score
    )

    if (
        best_score >= 0.35
        and score_margin >= 0.02
    ):
        return RoutingDecision(
            document_type=routing_types[
                best_index
            ],
            confidence=min(
                0.90,
                best_score
            ),
            method="Semantic",
            filter_applied=False
        )

    return RoutingDecision(
        document_type="Other",
        confidence=max(
            0.0,
            best_score
        ),
        method="Global",
        filter_applied=False
    )


def identify_retrieval_intent(
    query: str
) -> str:

    query_lower = normalise_routing_text(
        query
    )

    if any(
        phrase in query_lower
        for phrase in [
            "publication date",
            "when was",
            "when published",
            "published",
            "publication",
            "journal",
            "doi",
            "author"
        ]
    ):
        return "publication"

    if any(
        phrase in query_lower
        for phrase in [
            "key finding",
            "main finding",
            "findings",
            "results",
            "outcome",
            "conclusion",
            "what did the study find"
        ]
    ):
        return "findings"

    if any(
        phrase in query_lower
        for phrase in [
            "main purpose",
            "general point",
            "main point",
            "purpose",
            "objective",
            "aim",
            "what is this paper about"
        ]
    ):
        return "purpose"

    if any(
        phrase in query_lower
        for phrase in [
            "summarise",
            "summarize",
            "summary",
            "overview",
            "what does this document contain",
            "what does the document contain",
            "what is in this document",
            "what is this document about",
            "document contents",
            "contents of this document"
        ]
    ):
        return "summary"

    if any(
        phrase in query_lower
        for phrase in [
            "method",
            "methodology",
            "approach",
            "model used",
            "data used"
        ]
    ):
        return "methods"

    return "general"


def expand_retrieval_query(
    query: str
) -> str:

    intent = identify_retrieval_intent(
        query
    )

    expansions = {
        "publication": (
            "title page authors journal DOI publication date "
            "received accepted published"
        ),
        "findings": (
            "results findings conclusion discussion main outcomes "
            "the study found the results show"
        ),
        "purpose": (
            "abstract introduction objective aim purpose "
            "research question this study investigates"
        ),
        "summary": (
            "document overview purpose product information identifiers "
            "lot batch dates specifications quality requirements "
            "methods results findings compliance conclusion"
        ),
        "methods": (
            "methods methodology study design data model analysis "
            "materials and methods"
        ),
        "general": ""
    }

    expansion = expansions[
        intent
    ]

    if not expansion:
        return query.strip()

    return (
        f"{query.strip()}\n"
        f"Relevant document information: "
        f"{expansion}"
    )


def rerank_retrieved_chunks(
    query: str,
    retrieved_chunks: List[
        Tuple[DocumentChunk, float]
    ],
    top_k: int
) -> List[Tuple[DocumentChunk, float]]:

    intent = identify_retrieval_intent(
        query
    )

    section_terms = {
        "publication": [
            "doi",
            "journal",
            "published",
            "publication",
            "received",
            "accepted",
            "author"
        ],
        "findings": [
            "results",
            "findings",
            "conclusion",
            "we found",
            "we show",
            "our results",
            "the results show"
        ],
        "purpose": [
            "abstract",
            "objective",
            "purpose",
            "aim",
            "this study",
            "we investigate",
            "we examine"
        ],
        "summary": [
            "abstract",
            "introduction",
            "product",
            "article number",
            "lot number",
            "batch number",
            "manufacture",
            "expiration",
            "specification",
            "quality",
            "compliance",
            "results",
            "conclusion"
        ],
        "methods": [
            "methods",
            "methodology",
            "study design",
            "data",
            "model",
            "analysis"
        ],
        "general": []
    }

    relevant_terms = section_terms[
        intent
    ]

    reranked = []

    for chunk, semantic_score in retrieved_chunks:
        text_lower = chunk.text.lower()
        opening_text = text_lower[:1500]

        term_matches = sum(
            term in opening_text
            for term in relevant_terms
        )

        section_bonus = min(
            0.16,
            term_matches * 0.035
        )

        page_bonus = 0.0

        if (
            chunk.document_type
            == "Research Article"
        ):
            if (
                intent in [
                    "purpose",
                    "publication"
                ]
                and chunk.page_start <= 3
            ):
                page_bonus = 0.05

        reference_penalty = 0.0
        reference_opening = opening_text[:400]

        if (
            "references" in reference_opening
            or "bibliography" in reference_opening
        ):
            reference_penalty = 0.15

        relevance_score = (
            semantic_score
            + section_bonus
            + page_bonus
            - reference_penalty
        )

        relevance_score = max(
            0.0,
            min(
                1.0,
                relevance_score
            )
        )

        reranked.append(
            (
                chunk,
                relevance_score
            )
        )

    reranked.sort(
        key=lambda item: item[1],
        reverse=True
    )

    return reranked[
        :int(top_k)
    ]


def retrieve_for_query(
    query: str,
    top_k: int = DEFAULT_TOP_K,
    selected_document_type: str = "All Documents",
    auto_route: bool = True
) -> Tuple[
    List[Tuple[DocumentChunk, float]],
    RoutingDecision
]:

    if not query or not query.strip():
        raise ValueError(
            "Enter a question before searching."
        )

    retrieval_query = expand_retrieval_query(
        query
    )

    candidate_count = max(
        int(top_k) * 3,
        12
    )

    def search_and_rerank(
        document_type: Optional[str] = None
    ):
        candidates = rag_index.search(
            query=retrieval_query,
            top_k=candidate_count,
            document_type=document_type
        )

        return rerank_retrieved_chunks(
            query=query,
            retrieved_chunks=candidates,
            top_k=top_k
        )

    if (
        selected_document_type
        and selected_document_type
        != "All Documents"
    ):
        decision = RoutingDecision(
            document_type=(
                selected_document_type
            ),
            confidence=1.0,
            method="Manual",
            filter_applied=True
        )

        results = search_and_rerank(
            selected_document_type
        )

        return results, decision

    available_document_types = list(
        rag_index.document_type_indices.keys()
    )

    # When every processed file has the same document type,
    # the system can safely use that category directly.
    if len(available_document_types) == 1:
        only_document_type = (
            available_document_types[0]
        )

        decision = RoutingDecision(
            document_type=only_document_type,
            confidence=1.0,
            method="Single document type",
            filter_applied=True
        )

        results = search_and_rerank(
            only_document_type
        )

        return results, decision

    if auto_route:
        decision = route_query(
            query
        )

        route_available = (
            decision.document_type
            in rag_index.document_type_indices
        )

        if (
            decision.confidence >= 0.60
            and route_available
        ):
            decision.filter_applied = True

            results = search_and_rerank(
                decision.document_type
            )

            if results:
                return results, decision

    else:
        decision = RoutingDecision(
            document_type="All Documents",
            confidence=1.0,
            method="Global",
            filter_applied=False
        )

    decision = RoutingDecision(
        document_type="All Documents",
        confidence=(
            decision.confidence
            if auto_route
            else 1.0
        ),
        method=(
            decision.method
            if auto_route
            else "Global"
        ),
        filter_applied=False
    )

    results = search_and_rerank()

    return results, decision


print(
    "Query routing and retrieval functions ready."
)

Query routing and retrieval functions ready.


## 9. Prompting and Answer Generation

In [ ]:
def format_page_reference(
    page_start: int,
    page_end: int
) -> str:

    if page_start == page_end:
        return f"Page {page_start}"

    return f"Pages {page_start}–{page_end}"


def build_context(
    retrieved_chunks: List[
        Tuple[DocumentChunk, float]
    ]
) -> Tuple[str, List[Dict[str, Any]]]:

    context_sections = []
    sources = []

    for source_number, (chunk, score) in enumerate(
        retrieved_chunks,
        start=1
    ):
        page_reference = format_page_reference(
            chunk.page_start,
            chunk.page_end
        )

        context_sections.append(
            f"[Source {source_number}]\n"
            f"File: {chunk.file_name}\n"
            f"Document type: {chunk.document_type}\n"
            f"{page_reference}\n"
            f"Text:\n{chunk.text}"
        )

        sources.append(
            {
                "source_number": source_number,
                "file_name": chunk.file_name,
                "document_type": chunk.document_type,
                "page_reference": page_reference,
                "relevance": score,
                "chunk_id": chunk.chunk_id
            }
        )

    return (
        "\n\n".join(context_sections),
        sources
    )


def confidence_label(
    score: float
) -> str:

    if score >= 0.70:
        return "High"

    if score >= 0.40:
        return "Moderate"

    return "Low"


def count_available_chunks(
    selected_document_type: str,
    routing_decision: RoutingDecision
) -> int:

    if (
        selected_document_type
        and selected_document_type
        != "All Documents"
    ):
        return sum(
            chunk.document_type
            == selected_document_type
            for chunk in rag_index.chunks
        )

    if (
        routing_decision.filter_applied
        and routing_decision.document_type
        != "All Documents"
    ):
        return sum(
            chunk.document_type
            == routing_decision.document_type
            for chunk in rag_index.chunks
        )

    return len(
        rag_index.chunks
    )


def calculate_retrieval_confidence(
    retrieved_chunks: List[
        Tuple[DocumentChunk, float]
    ],
    selected_document_type: str,
    routing_decision: RoutingDecision
) -> Tuple[float, float, int]:

    relevance_scores = sorted(
        [
            score
            for _, score in retrieved_chunks
        ],
        reverse=True
    )

    strongest_scores = relevance_scores[
        :min(
            2,
            len(relevance_scores)
        )
    ]

    strongest_relevance = sum(
        strongest_scores
    ) / len(strongest_scores)

    available_chunks = max(
        1,
        count_available_chunks(
            selected_document_type,
            routing_decision
        )
    )

    evidence_coverage = min(
        1.0,
        len(retrieved_chunks)
        / available_chunks
    )

    relevance_component = min(
        0.55,
        max(0.0, strongest_relevance)
    )

    coverage_component = (
        0.30 * evidence_coverage
    )

    routing_component = 0.0

    if routing_decision.filter_applied:
        routing_component = (
            0.15
            * max(
                0.0,
                min(
                    1.0,
                    routing_decision.confidence
                )
            )
        )

    retrieval_confidence = min(
        1.0,
        relevance_component
        + coverage_component
        + routing_component
    )

    return (
        retrieval_confidence,
        evidence_coverage,
        available_chunks
    )


EXACT_FIELD_PATTERNS = {
    "process run id": [
        r"\bprocess\s+run\s+(?:id|identifier)\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "lot number": [
        r"\blot\s*(?:number|no\.?|#)\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "batch number": [
        r"\bbatch\s*(?:number|no\.?|#)\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "article number": [
        r"\barticle\s*(?:number|no\.?|#)\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "manufacture date": [
        r"\b(?:date\s+of\s+manufacture|manufacture\s+date|"
        r"manufacturing\s+date)"
        r"\s*[:\-]?\s*([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "release date": [
        r"\brelease\s+date\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "retest date": [
        r"\bretest\s+date\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ],
    "expiry date": [
        r"\b(?:expiry|expiration)\s+date\s*[:\-]?\s*"
        r"([A-Z0-9][A-Z0-9._/\-]{2,})"
    ]
}


EXACT_FIELD_QUERY_TERMS = {
    "process run id": [
        "process run id",
        "process run identifier",
        "run id"
    ],
    "lot number": [
        "lot number",
        "lot no",
        "lot #"
    ],
    "batch number": [
        "batch number",
        "batch no",
        "batch #"
    ],
    "article number": [
        "article number",
        "article no",
        "article #"
    ],
    "manufacture date": [
        "manufacture date",
        "manufacturing date",
        "date of manufacture",
        "manufactured"
    ],
    "release date": [
        "release date",
        "date of release"
    ],
    "retest date": [
        "retest date",
        "date of retest"
    ],
    "expiry date": [
        "expiry date",
        "expiration date",
        "expires",
        "expiry"
    ]
}


# Ensure certificate date questions use the existing metadata route.
for date_routing_term in [
    "manufacturing date",
    "release date",
    "retest date"
]:
    if date_routing_term not in (
        QUERY_ROUTING_KEYWORDS["Certificate of Quality"]
    ):
        QUERY_ROUTING_KEYWORDS[
            "Certificate of Quality"
        ].append(date_routing_term)


DATE_VALUE_PATTERN = (
    r"\b\d{1,4}[./\-]\d{1,2}[./\-]\d{1,4}\b"
)


DATE_FIELD_LABEL_PATTERNS = {
    "manufacture date": (
        r"\b(?:manufacturing\s+date|manufacture\s+date|"
        r"date\s+of\s+manufacture)\b"
    ),
    "release date": r"\brelease\s+date\b",
    "retest date": r"\bretest\s+date\b",
    "expiry date": (
        r"\b(?:expiry|expiration)\s+date\b"
    )
}


def extract_grouped_date_fields(
    text: str,
    requested_fields: List[str],
    source_number: int
) -> List[Dict[str, Any]]:
    """Pair grouped OCR date labels with grouped date values."""

    label_matches = []

    for field_name, pattern in (
        DATE_FIELD_LABEL_PATTERNS.items()
    ):
        match = re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        if match:
            label_matches.append(
                (match.start(), field_name)
            )

    label_matches.sort(
        key=lambda item: item[0]
    )

    # Grouped table recovery requires at least two labels.
    if len(label_matches) < 2:
        return []

    last_label_position = max(
        position
        for position, _ in label_matches
    )

    date_values = re.findall(
        DATE_VALUE_PATTERN,
        text[last_label_position:]
    )

    if len(date_values) < len(label_matches):
        return []

    recovered_fields = []

    for (_, field_name), value in zip(
        label_matches,
        date_values
    ):
        if field_name not in requested_fields:
            continue

        recovered_fields.append(
            {
                "field_name": field_name,
                "value": value,
                "source_number": source_number
            }
        )

    return recovered_fields


def extract_exact_fields(
    query: str,
    retrieved_chunks: List[
        Tuple[DocumentChunk, float]
    ]
) -> List[Dict[str, Any]]:
    """Extract requested identifiers directly from retrieved text."""

    query_lower = query.lower()

    requested_fields = [
        field_name
        for field_name, query_terms
        in EXACT_FIELD_QUERY_TERMS.items()
        if any(
            term in query_lower
            for term in query_terms
        )
    ]

    if not requested_fields:
        return []

    invalid_values = {
        "and",
        "are",
        "for",
        "is",
        "lot",
        "not",
        "number",
        "the",
        "unknown",
        "was",
        "were"
    }

    extracted_fields = []
    seen = set()

    # Exact values can sit in a different chunk on the same page.
    # Search the complete extracted page text whenever it is available.
    page_text_lookup = {}

    if "app_state" in globals():
        page_text_lookup = {
            (
                page.source_id,
                page.page_number
            ): page.text
            for page in app_state.get(
                "pages",
                []
            )
        }

    search_entries = []
    seen_pages = set()

    for source_number, (chunk, _) in enumerate(
        retrieved_chunks,
        start=1
    ):
        page_key = (
            chunk.source_id,
            chunk.page_start
        )

        if page_key in seen_pages:
            continue

        seen_pages.add(page_key)

        page_text = page_text_lookup.get(
            page_key,
            chunk.text
        )

        search_entries.append(
            (
                source_number,
                page_text
            )
        )

    for source_number, page_text in search_entries:
        searchable_text = re.sub(
            r"[ \t]+",
            " ",
            page_text
        )

        # OCR often reads a table as all labels followed by all values.
        # Recover that layout before applying the simpler inline patterns.
        requested_date_fields = [
            field_name
            for field_name in requested_fields
            if field_name in DATE_FIELD_LABEL_PATTERNS
        ]

        grouped_date_fields = extract_grouped_date_fields(
            text=searchable_text,
            requested_fields=requested_date_fields,
            source_number=source_number
        )

        grouped_field_names = {
            field["field_name"]
            for field in grouped_date_fields
        }

        for field in grouped_date_fields:
            unique_key = (
                field["field_name"],
                field["value"].lower()
            )

            if unique_key in seen:
                continue

            seen.add(unique_key)
            extracted_fields.append(field)

        for field_name in requested_fields:
            if field_name in grouped_field_names:
                continue

            for pattern in EXACT_FIELD_PATTERNS[
                field_name
            ]:
                for match in re.finditer(
                    pattern,
                    searchable_text,
                    flags=re.IGNORECASE
                ):
                    value = match.group(1).strip(
                        " .,:;"
                    )

                    if (
                        value.lower() in invalid_values
                        or not any(
                            character.isdigit()
                            for character in value
                        )
                    ):
                        continue

                    unique_key = (
                        field_name,
                        value.lower()
                    )

                    if unique_key in seen:
                        continue

                    seen.add(
                        unique_key
                    )

                    extracted_fields.append(
                        {
                            "field_name": field_name,
                            "value": value,
                            "source_number": source_number
                        }
                    )

    return extracted_fields


def format_exact_field_evidence(
    exact_fields: List[Dict[str, Any]]
) -> str:
    """Format exact values for the language-model prompt."""

    if not exact_fields:
        return "No separate exact-field evidence was requested."

    lines = [
        "Exact values extracted directly from the retrieved pages:"
    ]

    for field in exact_fields:
        lines.append(
            f"- {field['field_name'].title()}: "
            f"{field['value']} "
            f"[Source {field['source_number']}]"
        )

    return "\n".join(
        lines
    )


REGULATORY_REFERENCE_QUERY_TERMS = [
    "guidance",
    "guideline",
    "regulation",
    "regulatory reference",
    "standard",
    "requirement"
]


REGULATORY_REFERENCE_PATTERNS = [
    r"\b(?:EMEA|EMA)\s*/\s*\d{3,4}\s*/\s*\d{2,4}\b",
    (
        r"\b(?:EU\s+)?Regulation\s*"
        r"\(?(?:EC|EU)\)?\s*No\.?\s*\d+/\d+\b"
    )
]


def extract_regulatory_references(
    query: str,
    retrieved_chunks: List[
        Tuple[DocumentChunk, float]
    ]
) -> List[Dict[str, Any]]:
    """Extract requested guidance codes from the strongest source."""

    query_lower = query.lower()

    if not any(
        term in query_lower
        for term in REGULATORY_REFERENCE_QUERY_TERMS
    ):
        return []

    references = []
    seen = set()

    # Stop after the first retrieved source containing a reference.
    # This keeps references tied to the strongest matching evidence.
    for source_number, (chunk, _) in enumerate(
        retrieved_chunks,
        start=1
    ):
        source_references = []

        for pattern in REGULATORY_REFERENCE_PATTERNS:
            for match in re.finditer(
                pattern,
                chunk.text,
                flags=re.IGNORECASE
            ):
                value = re.sub(
                    r"\s*/\s*",
                    "/",
                    match.group(0)
                )

                value = re.sub(
                    r"\s+",
                    " ",
                    value
                ).strip(" .,:;")

                if value.lower() in seen:
                    continue

                seen.add(value.lower())
                source_references.append(
                    {
                        "value": value,
                        "source_number": source_number
                    }
                )

        if source_references:
            references.extend(source_references)
            break

    return references


def format_regulatory_reference_evidence(
    regulatory_references: List[Dict[str, Any]]
) -> str:
    """Format exact regulatory references for the prompt."""

    if not regulatory_references:
        return "No separate regulatory reference was requested."

    lines = [
        "Regulatory references extracted from the strongest source:"
    ]

    for reference in regulatory_references:
        lines.append(
            f"- {reference['value']} "
            f"[Source {reference['source_number']}]"
        )

    return "\n".join(lines)


def build_exact_field_answer(
    exact_fields: List[Dict[str, Any]]
) -> str:
    """Create a reliable answer when the model omits exact values."""

    grouped_fields = {}

    for field in exact_fields:
        grouped_fields.setdefault(
            field["field_name"],
            []
        ).append(
            field
        )

    answer_lines = [
        "The requested values are:"
    ]

    for field_name, fields in grouped_fields.items():
        formatted_values = [
            f"**{field['value']}** "
            f"[Source {field['source_number']}]"
            for field in fields
        ]

        if len(formatted_values) == 1:
            answer_lines.append(
                f"- **{field_name.title()}:** "
                f"{formatted_values[0]}"
            )
        else:
            values_text = ", ".join(
                formatted_values[:-1]
            )

            if values_text:
                values_text += (
                    f" and {formatted_values[-1]}"
                )
            else:
                values_text = formatted_values[-1]

            answer_lines.append(
                f"- **{field_name.title()}:** "
                f"{values_text}"
            )

    return "\n".join(
        answer_lines
    )


def clean_generated_answer(
    answer: str
) -> str:
    """Remove generated bibliographies and redundant citation lines."""

    if not answer:
        return ""

    reference_patterns = [
        (
            r"\n\s*(?:#{1,6}\s*)?"
            r"references?\s*:?\s*\n"
        ),
        (
            r"\n\s*(?:#{1,6}\s*)?"
            r"bibliography\s*:?\s*\n"
        ),
        (
            r"\n\s*(?:#{1,6}\s*)?"
            r"reference list[^\n]*\n"
        ),
        (
            r"\n\s*(?:#{1,6}\s*)?"
            r"list of references[^\n]*\n"
        )
    ]

    cut_positions = []

    for pattern in reference_patterns:
        match = re.search(
            pattern,
            answer,
            flags=re.IGNORECASE
        )

        if match:
            cut_positions.append(
                match.start()
            )

    if cut_positions:
        answer = answer[
            :min(cut_positions)
        ]

    answer = re.sub(
        r"(?im)^\s*(?:"
        r"\[Source\s+\d+\]\s*"
        r")+\s*$",
        "",
        answer
    )

    answer = re.sub(
        r"(?im)^\s*(?:sources?|citations?)\s*:\s*(?:"
        r"\[Source\s+\d+\]\s*"
        r")+\s*$",
        "",
        answer
    )

    answer = re.sub(
        r"(\[Source\s+\d+\])\[\d+\]",
        r"\1",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"\n{3,}",
        "\n\n",
        answer
    )

    return answer.strip()


def answer_question(
    query: str,
    top_k: int = DEFAULT_TOP_K,
    selected_document_type: str = "All Documents",
    auto_route: bool = True
) -> Dict[str, Any]:

    if not rag_index.is_ready:
        return {
            "answer": (
                "Upload and process at least one PDF before "
                "asking a question."
            ),
            "sources": [],
            "confidence": 0.0,
            "confidence_label": "Low",
            "coverage": 0.0,
            "available_chunks": 0,
            "chunks_used": 0,
            "route": "Not available",
            "route_method": "None",
            "filter_applied": False,
            "response_seconds": 0.0
        }

    start_time = time.perf_counter()

    retrieved_chunks, routing_decision = (
        retrieve_for_query(
            query=query,
            top_k=top_k,
            selected_document_type=(
                selected_document_type
            ),
            auto_route=auto_route
        )
    )

    if not retrieved_chunks:
        return {
            "answer": (
                "I could not find relevant information in the "
                "processed documents."
            ),
            "sources": [],
            "confidence": 0.0,
            "confidence_label": "Low",
            "coverage": 0.0,
            "available_chunks": 0,
            "chunks_used": 0,
            "route": (
                routing_decision.document_type
            ),
            "route_method": (
                routing_decision.method
            ),
            "filter_applied": False,
            "response_seconds": round(
                time.perf_counter()
                - start_time,
                2
            )
        }

    context, sources = build_context(
        retrieved_chunks
    )

    exact_fields = extract_exact_fields(
        query=query,
        retrieved_chunks=retrieved_chunks
    )

    exact_field_evidence = format_exact_field_evidence(
        exact_fields
    )

    regulatory_references = extract_regulatory_references(
        query=query,
        retrieved_chunks=retrieved_chunks
    )

    regulatory_reference_evidence = (
        format_regulatory_reference_evidence(
            regulatory_references
        )
    )

    system_prompt = """
You are a document research assistant specialising in scientific
and pharmaceutical documents.

Answer the question using only the supplied document context.

Requirements:
- Do not use outside knowledge.
- If the answer is not directly supported by the context, say so clearly.
- Cite supporting information using [Source 1], [Source 2], and so on.
- Use only the source numbers supplied in the document context.
- Keep the answer focused, factual, and concise.
- Answer every distinct part of the user's question.
- Prioritise passages that directly address the question.
- Do not introduce unrelated declarations, tests, or substances.
- Do not invent document details, page numbers, dates, or measurements.
- Do not reproduce the document's bibliography or reference list.
- Do not create a References, Reference, or Bibliography section.
- Do not list academic works cited inside the uploaded document.
- Do not treat a cited paper's publication date as the publication
  date of the uploaded document.
- Use citations inside relevant sentences.
- Do not finish with a separate line containing only source citations.
- When the question requests a lot number, batch number, article number,
  or date, state the exact value rather than replacing it with a citation.
- When the question asks which guidance, regulation, or standard applies,
  copy its exact title or reference code from the supplied evidence.
""".strip()

    user_prompt = f"""
Question:
{query}

Document context:
{context}

{exact_field_evidence}

{regulatory_reference_evidence}

Answer the question using only the supplied context. Include short
inline citations in the format [Source N]. Do not add a bibliography,
reference list, or citation-only line after the answer. Copy requested
identifiers, dates, and regulatory references exactly as they appear
in the evidence. Address each requested point and omit unrelated topics.
""".strip()

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    answer = generate_text(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        max_new_tokens=400
    )

    answer = clean_generated_answer(
        answer
    )

    missing_exact_values = [
        field
        for field in exact_fields
        if field["value"].lower()
        not in answer.lower()
    ]

    if exact_fields and missing_exact_values:
        answer = build_exact_field_answer(
            exact_fields
        )

    missing_regulatory_references = [
        reference
        for reference in regulatory_references
        if reference["value"].lower()
        not in answer.lower()
    ]

    if missing_regulatory_references:
        reference_lines = [
            (
                f"The cited guidance or regulatory reference is "
                f"**{reference['value']}** "
                f"[Source {reference['source_number']}]."
            )
            for reference in missing_regulatory_references
        ]

        answer = (
            f"{answer.rstrip()}\n\n"
            f"{chr(10).join(reference_lines)}"
        ).strip()

    if not answer:
        answer = (
            "The retrieved context did not contain enough "
            "information to answer the question."
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    (
        retrieval_confidence,
        evidence_coverage,
        available_chunks
    ) = calculate_retrieval_confidence(
        retrieved_chunks=retrieved_chunks,
        selected_document_type=selected_document_type,
        routing_decision=routing_decision
    )

    response_seconds = round(
        time.perf_counter() - start_time,
        2
    )

    return {
        "answer": answer,
        "sources": sources,
        "confidence": retrieval_confidence,
        "confidence_label": confidence_label(
            retrieval_confidence
        ),
        "coverage": evidence_coverage,
        "available_chunks": available_chunks,
        "chunks_used": len(
            retrieved_chunks
        ),
        "route": (
            routing_decision.document_type
        ),
        "route_method": (
            routing_decision.method
        ),
        "filter_applied": (
            routing_decision.filter_applied
        ),
        "response_seconds": response_seconds
    }


def format_answer(
    result: Dict[str, Any]
) -> str:

    # Remove inline labels because sources are displayed separately below.
    answer = re.sub(
        (
            r"\s*\[Source\s+\d+"
            r"(?:\s*(?:,|&|and)\s*"
            r"(?:Source\s+)?\d+)*\]"
        ),
        "",
        result["answer"],
        flags=re.IGNORECASE
    ).strip()

    if not result["sources"]:
        return answer

    source_lines = []

    for source in result["sources"]:
        source_lines.append(
            f"- **[Source "
            f"{source['source_number']}]** "
            f"{source['file_name']} — "
            f"{source['document_type']}, "
            f"{source['page_reference']}"
        )

    details = (
        f"**Retrieval confidence:** "
        f"{result['confidence']:.0%} "
        f"({result['confidence_label']})  \n"
        f"**Evidence coverage:** "
        f"{result['chunks_used']} of "
        f"{result['available_chunks']} "
        f"available chunks  \n"
        f"**Chunks used:** "
        f"{result['chunks_used']}  \n"
        f"**Route:** "
        f"{result['route']} "
        f"({result['route_method']})  \n"
        f"**Response time:** "
        f"{result['response_seconds']:.2f} "
        f"seconds"
    )

    return (
        f"{answer}\n\n"
        f"### Sources\n"
        f"{chr(10).join(source_lines)}\n\n"
        f"### Retrieval details\n"
        f"{details}"
    )


# Preserve the standard RAG function for non-metadata questions
answer_rag_question = answer_question


PAGE_COUNT_PATTERNS = (
    r"\bhow many pages\b",
    r"\bnumber of pages\b",
    r"\bpage count\b",
    r"\btotal pages\b"
)


def answer_question(
    query: str,
    top_k: int = DEFAULT_TOP_K,
    selected_document_type: str = "All Documents",
    auto_route: bool = True
) -> Dict[str, Any]:

    normalised_query = re.sub(
        r"\s+",
        " ",
        (query or "").lower()
    ).strip()

    is_page_count_question = any(
        re.search(pattern, normalised_query)
        for pattern in PAGE_COUNT_PATTERNS
    )

    if is_page_count_question:
        summaries = app_state.get("summaries", [])

        if not summaries:
            return {
                "answer": (
                    "Upload and process at least one PDF before "
                    "asking for its page count."
                ),
                "sources": []
            }

        total_pages = sum(
            summary.total_pages
            for summary in summaries
        )

        if len(summaries) == 1:
            summary = summaries[0]
            page_word = (
                "page"
                if summary.total_pages == 1
                else "pages"
            )

            metadata_answer = (
                f"The uploaded document "
                f"**{summary.file_name}** contains "
                f"**{summary.total_pages} {page_word}**."
            )

        else:
            page_word = (
                "page"
                if total_pages == 1
                else "pages"
            )

            file_lines = [
                (
                    f"- **{summary.file_name}:** "
                    f"{summary.total_pages} "
                    f"{'page' if summary.total_pages == 1 else 'pages'}"
                )
                for summary in summaries
            ]

            metadata_answer = (
                f"The uploaded documents contain "
                f"**{total_pages} {page_word} in total**.\n\n"
                f"{chr(10).join(file_lines)}"
            )

        return {
            "answer": metadata_answer,
            "sources": []
        }

    return answer_rag_question(
        query=query,
        top_k=top_k,
        selected_document_type=selected_document_type,
        auto_route=auto_route
    )


print(
    "Prompting and answer-generation functions ready."
)


Prompting and answer-generation functions ready.


## 10. Pipeline Integration

In [ ]:
app_state = {
    "pages": [],
    "documents": [],
    "chunks": [],
    "summaries": [],
    "processing_seconds": 0.0
}


def format_processing_summary(
    summaries: List[FileSummary],
    logical_documents: List[LogicalDocument],
    chunks: List[DocumentChunk],
    total_seconds: float
) -> str:

    lines = [
        "### Processing complete",
        "",
        "| File | Pages | Digital | OCR | Documents | Chunks |",
        "|---|---:|---:|---:|---:|---:|"
    ]

    for summary in summaries:
        safe_name = summary.file_name.replace("|", "\\|")

        lines.append(
            f"| {safe_name} "
            f"| {summary.total_pages} "
            f"| {summary.digital_pages} "
            f"| {summary.ocr_pages} "
            f"| {summary.document_count} "
            f"| {summary.chunk_count} |"
        )

    type_counts = {}

    for document in logical_documents:
        type_counts[document.document_type] = (
            type_counts.get(document.document_type, 0) + 1
        )

    type_summary = ", ".join(
        f"{document_type}: {count}"
        for document_type, count in sorted(type_counts.items())
    )

    lines.extend(
        [
            "",
            f"**Files processed:** {len(summaries)}  ",
            f"**Total pages:** "
            f"{sum(summary.total_pages for summary in summaries)}  ",
            f"**OCR pages:** "
            f"{sum(summary.ocr_pages for summary in summaries)}  ",
            f"**Chunks indexed:** {len(chunks)}  ",
            f"**Processing time:** {total_seconds:.2f} seconds  ",
            f"**Document types:** {type_summary or 'Other'}"
        ]
    )

    return "\n".join(lines)


def process_uploaded_documents(
    uploaded_files: Any
):
    if not uploaded_files:
        return (
            "Upload at least one PDF before processing.",
            gr.Dropdown(
                choices=["All Documents"],
                value="All Documents"
            )
        )

    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]

    start_time = time.perf_counter()

    try:
        pages, summaries = extract_multiple_pdfs(
            uploaded_files
        )

        logical_documents = segment_all_documents(
            pages,
            summaries
        )

        chunks = create_document_chunks(
            logical_documents,
            summaries
        )

        if not chunks:
            raise ValueError(
                "No readable text was found in the uploaded PDFs."
            )

        rag_index.build(chunks)

        total_seconds = round(
            time.perf_counter() - start_time,
            2
        )

        app_state.update(
            {
                "pages": pages,
                "documents": logical_documents,
                "chunks": chunks,
                "summaries": summaries,
                "processing_seconds": total_seconds
            }
        )

        available_types = sorted(
            rag_index.document_type_indices.keys()
        )

        filter_choices = [
            "All Documents",
            *available_types
        ]

        status = format_processing_summary(
            summaries=summaries,
            logical_documents=logical_documents,
            chunks=chunks,
            total_seconds=total_seconds
        )

        return (
            status,
            gr.Dropdown(
                choices=filter_choices,
                value="All Documents"
            )
        )

    except Exception as error:
        rag_index.clear()

        app_state.update(
            {
                "pages": [],
                "documents": [],
                "chunks": [],
                "summaries": [],
                "processing_seconds": 0.0
            }
        )

        return (
            f"### Processing failed\n\n{str(error)}",
            gr.Dropdown(
                choices=["All Documents"],
                value="All Documents"
            )
        )


def clear_processed_documents():
    rag_index.clear()

    app_state.update(
        {
            "pages": [],
            "documents": [],
            "chunks": [],
            "summaries": [],
            "processing_seconds": 0.0
        }
    )

    return (
        None,
        "No documents have been processed.",
        gr.Dropdown(
            choices=["All Documents"],
            value="All Documents"
        ),
        []
    )


print("Document-processing pipeline ready.")

Document-processing pipeline ready.


## 11. Gradio Interface

In [ ]:
MODE_DEFAULT_QUESTIONS = {
    "Ask a Question": "",
    "Key Findings": (
        "Identify the five most important findings across the processed "
        "documents. Cover different document types where possible. Use "
        "five concise numbered points, cite the supporting sources, and "
        "keep the complete answer below 170 words. Do not reproduce long "
        "lists of part numbers, test results, or document contents."
    ),
    "Summarise Document": (
        "Provide a concise structured summary of the processed documents. "
        "Explain their purpose, the document types present, the most "
        "important product, quality, compliance, and supplier details, "
        "and any overall conclusion. Cite supporting sources and keep "
        "the complete answer below 180 words."
    )
}


STATUS_READY = """
<div class="assistant-status ready-status">
    <span class="status-light"></span>
    Ready for a question
</div>
"""


STATUS_WORKING = """
<div class="assistant-status working-status">
    <span class="status-light"></span>
    Searching the documents and generating an answer...
</div>
"""


STATUS_ERROR = """
<div class="assistant-status error-status">
    <span class="status-light"></span>
    The request could not be completed
</div>
"""


def prepare_mode_query(
    question: str,
    analysis_mode: str
) -> str:

    question = (question or "").strip()
    analysis_mode = analysis_mode or "Ask a Question"

    if analysis_mode == "Ask a Question":
        return question

    default_question = MODE_DEFAULT_QUESTIONS.get(
        analysis_mode,
        ""
    )

    if not question:
        return default_question

    if analysis_mode == "Key Findings":
        return (
            f"{default_question}\n\n"
            f"Pay particular attention to: {question}"
        )

    return (
        f"{default_question}\n\n"
        f"Focus the summary on: {question}"
    )


def format_display_question(
    question: str,
    analysis_mode: str
) -> str:

    question = (question or "").strip()
    analysis_mode = analysis_mode or "Ask a Question"

    if analysis_mode == "Ask a Question":
        return question

    if not question:
        return analysis_mode

    if analysis_mode == "Key Findings":
        return f"Key Findings — focus on: {question}"

    return f"Summarise Document — focus on: {question}"


def preview_first_pdf(uploaded_files):

    if not uploaded_files:
        return None

    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]

    return resolve_file_path(uploaded_files[0])


def process_documents_for_ui(uploaded_files):

    status, updated_filter = process_uploaded_documents(
        uploaded_files
    )

    if (
        isinstance(status, str)
        and status.startswith("### Processing failed")
    ):
        print(status)

        return (
            "### Processing could not be completed\n\n"
            "Check the notebook output for technical details.",
            updated_filter
        )

    return status, updated_filter


def run_chat_turn(
    question: str,
    history: Optional[List[Dict[str, Any]]],
    analysis_mode: str,
    selected_document_type: str,
    auto_route: bool,
    top_k: int,
    scroll_value: float
):

    history = list(history or [])
    analysis_mode = analysis_mode or "Ask a Question"

    prepared_question = prepare_mode_query(
        question,
        analysis_mode
    )

    if not prepared_question:
        yield (
            "",
            history,
            STATUS_READY,
            gr.skip()
        )
        return

    display_question = format_display_question(
        question,
        analysis_mode
    )

    history.append(
        {
            "role": "user",
            "content": display_question
        }
    )

    # Reserve the assistant position below the new question.
    history.append(
        {
            "role": "assistant",
            "content": "…"
        }
    )

    # Position the new exchange while the answer is generated.
    yield (
        "",
        history,
        STATUS_WORKING,
        time.time()
    )

    selected_document_type = (
        selected_document_type or "All Documents"
    )

    retrieval_count = int(
        top_k if top_k is not None else DEFAULT_TOP_K
    )

    use_auto_route = bool(auto_route)

    if analysis_mode == "Key Findings":
        retrieval_count = max(
            retrieval_count,
            6
        )
        use_auto_route = False

    elif analysis_mode == "Summarise Document":
        retrieval_count = max(
            retrieval_count,
            8
        )
        use_auto_route = False

    try:
        result = answer_question(
            query=prepared_question,
            top_k=retrieval_count,
            selected_document_type=selected_document_type,
            auto_route=use_auto_route
        )

        response = format_answer(result)
        final_status = STATUS_READY

    except Exception as error:
        print(
            "Question-processing error:",
            repr(error)
        )

        response = (
            "The question could not be processed. "
            "Check the notebook output for technical details."
        )

        final_status = STATUS_ERROR

    # Replace the temporary assistant message in the same position.
    history[-1] = {
        "role": "assistant",
        "content": response
    }

    # Position the completed answer underneath the newest question.
    yield (
        "",
        history,
        final_status,
        time.time()
    )


CUSTOM_CSS = """
body {
    background: #eef1ee;
}

body.chat-modal-open {
    overflow: hidden !important;
}

.gradio-container {
    max-width: 1500px !important;
    margin: 0 auto !important;
    padding: 12px !important;
    background: #eef1ee !important;
    font-family:
        "Avenir Next",
        "Trebuchet MS",
        Arial,
        sans-serif;
}

.app-header {
    display: flex;
    align-items: center;
    padding: 16px 22px;
    margin-bottom: 14px;
    border: 1px solid #c7d8d3;
    border-radius: 24px 24px 7px 24px;
    background:
        linear-gradient(
            125deg,
            #dff2ee 0%,
            #edf5ed 58%,
            #fff2d9 100%
        );
    box-shadow:
        0 9px 24px
        rgba(30, 57, 67, 0.10);
}

.brand-block {
    display: flex;
    align-items: center;
    gap: 15px;
}

.brand-mark {
    display: flex;
    align-items: center;
    justify-content: center;
    width: 50px;
    height: 50px;
    flex-shrink: 0;
    border-radius: 16px 16px 5px 16px;
    color: #ffffff;
    background: #0f6b63;
    font-family:
        Georgia,
        "Times New Roman",
        serif;
    font-size: 19px;
    font-weight: 700;
    letter-spacing: -1px;
    box-shadow:
        0 6px 14px
        rgba(15, 107, 99, 0.20);
}

.eyebrow {
    display: block;
    margin-bottom: 2px;
    color: #8b5e34;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 1.5px;
}

.app-header h1 {
    margin: 0;
    color: #17324d !important;
    font-family:
        Georgia,
        "Times New Roman",
        serif;
    font-size: 28px;
    font-weight: 700;
    letter-spacing: -0.4px;
}

.app-header p {
    margin: 4px 0 0;
    color: #405c68 !important;
    font-size: 14px;
}

.panel {
    padding: 15px !important;
    border: 1.5px solid #c8d5d1 !important;
    border-radius: 17px 17px 6px 17px !important;
    background: #ffffff !important;
    box-shadow:
        0 8px 24px
        rgba(39, 59, 64, 0.10);
}

.panel-title h3 {
    margin: 0 0 8px !important;
    padding-left: 9px;
    border-left: 4px solid #dc8d4a;
    color: #17324d;
    font-family:
        Georgia,
        "Times New Roman",
        serif;
    font-size: 19px;
}

#pdf-upload {
    max-height: 120px;
    overflow-y: auto;
}

#processing-status {
    height: 145px;
    overflow: auto;
    padding: 10px;
    border: 1px solid #d7e2de;
    border-radius: 11px;
    background: #f6faf8;
}

#processing-status h3 {
    margin-top: 0;
}

#processing-status table {
    font-size: 11px;
}

#research-chat {
    position: relative;
    border: 1px solid #d0ded9;
    border-radius: 14px 14px 5px 14px;
    background: #ffffff;
    transition:
        width 0.2s ease,
        height 0.2s ease;
}

#research-chat.chat-expanded {
    position: fixed !important;
    inset: 16px !important;
    z-index: 99990 !important;
    width: calc(100vw - 32px) !important;
    height: calc(100vh - 32px) !important;
    max-width: none !important;
    max-height: none !important;
    margin: 0 !important;
    padding: 10px !important;
    border: 2px solid #0f6b63 !important;
    border-radius: 18px !important;
    background: #ffffff !important;
    box-shadow:
        0 24px 70px
        rgba(13, 40, 48, 0.32) !important;
}

#research-chat.chat-expanded > div {
    height: 100% !important;
    max-height: none !important;
}

#research-chat.chat-expanded .wrap {
    height: 100% !important;
    max-height: none !important;
}

#research-chat.chat-expanded .message-wrap {
    max-width: 95% !important;
}

#expand-chat-button.chat-button-expanded {
    position: fixed !important;
    top: 27px !important;
    right: 34px !important;
    z-index: 100000 !important;

    width: 42px !important;
    min-width: 42px !important;
    max-width: 42px !important;
    height: 42px !important;

    padding: 0 !important;

    border-radius: 50% !important;

    font-size: 24px !important;
    line-height: 1 !important;

    box-shadow:
        0 5px 18px
        rgba(13, 40, 48, 0.20) !important;
}

#process-button,
#ask-button {
    border: none !important;
    color: #ffffff !important;
    background: #0f6b63 !important;
    font-weight: 750 !important;
}

#process-button:hover,
#ask-button:hover {
    background: #0b5751 !important;
}

#expand-chat-button {
    border: 1px solid #b9cfca !important;
    color: #17324d !important;
    background: #edf5f2 !important;
    font-weight: 700 !important;
}

#expand-chat-button:hover {
    background: #dcece7 !important;
}

.compact-row {
    gap: 9px !important;
}

#assistant-status {
    min-height: 34px;
}

.assistant-status {
    display: flex;
    align-items: center;
    gap: 9px;
    min-height: 34px;
    padding: 7px 12px;
    border-radius: 10px;
    font-size: 12px;
    font-weight: 700;
}

.status-light {
    display: inline-block;
    width: 9px;
    height: 9px;
    flex-shrink: 0;
    border-radius: 50%;
}

.ready-status {
    color: #315c52;
    border: 1px solid #c8ded7;
    background: #eef8f4;
}

.ready-status .status-light {
    background: #3d9d7b;
}

.working-status {
    color: #6f4b25;
    border: 1px solid #ead2ac;
    background: #fff6e8;
}

.working-status .status-light {
    background: #dc8d4a;
    animation: status-pulse 1.15s infinite;
}

.error-status {
    color: #7b3434;
    border: 1px solid #ebc4c4;
    background: #fff1f1;
}

.error-status .status-light {
    background: #bd4f4f;
}

@keyframes status-pulse {
    0% {
        opacity: 0.35;
        transform: scale(0.82);
    }

    50% {
        opacity: 1;
        transform: scale(1.25);
    }

    100% {
        opacity: 0.35;
        transform: scale(0.82);
    }
}

footer {
    display: none !important;
}

@media (max-width: 900px) {
    .app-header {
        align-items: flex-start;
    }

    .app-header h1 {
        font-size: 24px;
    }

    #research-chat.chat-expanded {
        inset: 8px !important;
        width: calc(100vw - 16px) !important;
        height: calc(100vh - 16px) !important;
    }

    #expand-chat-button.chat-button-expanded {
        top: 18px !important;
        right: 20px !important;
    }
}
"""

TOGGLE_CHAT_JAVASCRIPT = """
() => {
    const chat = document.getElementById(
        "research-chat"
    );

    const button = document.getElementById(
        "expand-chat-button"
    );

    if (!chat || !button) {
        return;
    }

    const isExpanded = chat.classList.toggle(
        "chat-expanded"
    );

    button.classList.toggle(
        "chat-button-expanded",
        isExpanded
    );

    document.body.classList.toggle(
        "chat-modal-open",
        isExpanded
    );

    const buttonText = button.querySelector(
        ".button-text"
    );

    if (buttonText) {
        buttonText.textContent = (
            isExpanded
            ? "×"
            : "⛶ Expand Chat"
        );
    } else {
        button.textContent = (
            isExpanded
            ? "×"
            : "⛶ Expand Chat"
        );
    }
}
"""


SCROLL_TO_NEW_ANSWER_JAVASCRIPT = """
(value) => {
    window.setTimeout(() => {
        const chat = document.getElementById(
            "research-chat"
        );

        if (!chat) {
            return value;
        }

        const conversation = chat.querySelector(
            '[aria-label="chatbot conversation"]'
        );

        if (!conversation) {
            return value;
        }

        conversation.style.overflowAnchor = "none";

        const userMessages = conversation.querySelectorAll(
            '[data-testid="user"]'
        );

        if (!userMessages.length) {
            return value;
        }

        const latestUserMessage = userMessages[
            userMessages.length - 1
        ];

        const latestUserRow = (
            latestUserMessage.closest(".message-row")
            || latestUserMessage
        );

        const conversationBox = (
            conversation.getBoundingClientRect()
        );

        const questionBox = (
            latestUserRow.getBoundingClientRect()
        );

        const targetPosition = (
            conversation.scrollTop
            + questionBox.top
            - conversationBox.top
            - 12
        );

        conversation.scrollTo({
            top: Math.max(0, targetPosition),
            behavior: "auto"
        });
    }, 160);

    return value;
}
"""


APP_JAVASCRIPT = """
() => {
    document.addEventListener(
        "keydown",
        (event) => {
            if (event.key !== "Escape") {
                return;
            }

            const chat = document.getElementById(
                "research-chat"
            );

            const button = document.getElementById(
                "expand-chat-button"
            );

            if (chat) {
                chat.classList.remove(
                    "chat-expanded"
                );
            }

            if (button) {
                button.classList.remove(
                    "chat-button-expanded"
                );
            }

            document.body.classList.remove(
                "chat-modal-open"
            );
        }
    );
}
"""


# Close the previous server when this cell is rerun.
try:
    demo.close()
except (NameError, AttributeError):
    pass


with gr.Blocks(
    title="PharmaDoc Intelligence",
    fill_width=True
) as demo:

    gr.HTML(
        """
        <div class="app-header">
            <div class="brand-block">
                <div class="brand-mark">PD</div>

                <div>
                    <span class="eyebrow">
                        OPEN-SOURCE DOCUMENT Q&amp;A
                    </span>

                    <h1>PharmaDoc Intelligence</h1>

                    <p>
                        Source-grounded answers from digital and
                        scanned pharmaceutical documents
                    </p>
                </div>
            </div>
        </div>
        """
    )

    with gr.Row(
        equal_height=True,
        elem_classes="main-layout"
    ):

        with gr.Column(
            scale=1,
            min_width=340,
            elem_classes="panel"
        ):
            gr.Markdown(
                "### Document Workspace",
                elem_classes="panel-title"
            )

            pdf_upload = gr.File(
                label="Upload one or more PDFs",
                file_types=[".pdf"],
                file_count="multiple",
                type="filepath",
                elem_id="pdf-upload"
)


            with gr.Row(elem_classes="compact-row"):
                process_button = gr.Button(
                    "Process Documents",
                    variant="primary",
                    elem_id="process-button"
                )

                clear_documents_button = gr.Button(
                    "Clear Documents"
                )

            processing_status = gr.Markdown(
                "No documents have been processed.",
                elem_id="processing-status"
            )

            with gr.Accordion(
                "PDF preview",
                open=False
            ):
                pdf_preview = PDF(
                    label="First uploaded PDF",
                    height=300,
                    interactive=False
                )

            analysis_mode = gr.Dropdown(
                choices=[
                    "Ask a Question",
                    "Key Findings",
                    "Summarise Document"
                ],
                value="Ask a Question",
                label="Analysis mode"
            )

            with gr.Accordion(
                "Advanced retrieval",
                open=False
            ):
                document_type_filter = gr.Dropdown(
                    choices=["All Documents"],
                    value="All Documents",
                    label="Document type"
                )

                auto_route_checkbox = gr.Checkbox(
                    value=True,
                    label="Automatically route questions"
                )

                top_k_slider = gr.Slider(
                    minimum=2,
                    maximum=8,
                    value=DEFAULT_TOP_K,
                    step=1,
                    label="Chunks to retrieve"
                )

        with gr.Column(
            scale=2,
            min_width=560,
            elem_classes="panel"
        ):
            gr.Markdown(
                "### Document Q&A",
                elem_classes="panel-title"
            )

            chatbot = gr.Chatbot(
                value=[],
                height=335,
                resizable=True,
                autoscroll=False,
                buttons=[
                    "copy",
                    "copy_all"
                ],
                layout="bubble",
                placeholder=(
                    "Process one or more PDFs, then ask a "
                    "question about their contents."
                ),
                elem_id="research-chat"
            )

            question_input = gr.Textbox(
                label="Question or optional focus",
                placeholder=(
                    "Enter a question, or leave blank when using "
                    "Key Findings or Summarise Document..."
                ),
                lines=1,
                max_lines=2
            )

            assistant_status = gr.HTML(
                value=STATUS_READY,
                elem_id="assistant-status"
            )

            scroll_trigger = gr.Number(
                value=0,
                visible=False
            )

            with gr.Row(elem_classes="compact-row"):
                ask_button = gr.Button(
                    "Ask Assistant",
                    variant="primary",
                    elem_id="ask-button"
                )

                expand_chat_button = gr.Button(
                    "⛶ Expand Chat",
                    elem_id="expand-chat-button"
                )

                clear_chat_button = gr.Button(
                    "Clear Chat"
                )

    pdf_upload.change(
        fn=preview_first_pdf,
        inputs=pdf_upload,
        outputs=pdf_preview
    )

    process_event = process_button.click(
        fn=process_documents_for_ui,
        inputs=pdf_upload,
        outputs=[
            processing_status,
            document_type_filter
        ],
        show_progress="full",
        show_progress_on=processing_status
    )

    process_event.then(
        fn=lambda: (
            [],
            STATUS_READY
        ),
        outputs=[
            chatbot,
            assistant_status
        ]
    )

    ask_button.click(
        fn=run_chat_turn,
        inputs=[
            question_input,
            chatbot,
            analysis_mode,
            document_type_filter,
            auto_route_checkbox,
            top_k_slider,
            scroll_trigger
        ],
        outputs=[
            question_input,
            chatbot,
            assistant_status,
            scroll_trigger
        ],
        show_progress="hidden"
    )

    question_input.submit(
        fn=run_chat_turn,
        inputs=[
            question_input,
            chatbot,
            analysis_mode,
            document_type_filter,
            auto_route_checkbox,
            top_k_slider,
            scroll_trigger
        ],
        outputs=[
            question_input,
            chatbot,
            assistant_status,
            scroll_trigger
        ],
        show_progress="hidden"
    )

    scroll_trigger.change(
        fn=None,
        inputs=scroll_trigger,
        outputs=None,
        js=SCROLL_TO_NEW_ANSWER_JAVASCRIPT,
        queue=False
    )

    expand_chat_button.click(
        fn=None,
        inputs=None,
        outputs=None,
        js=TOGGLE_CHAT_JAVASCRIPT,
        queue=False
    )

    clear_chat_button.click(
        fn=lambda: (
            [],
            "",
            STATUS_READY
        ),
        outputs=[
            chatbot,
            question_input,
            assistant_status
        ]
    )

    clear_documents_event = clear_documents_button.click(
        fn=clear_processed_documents,
        outputs=[
            pdf_upload,
            processing_status,
            document_type_filter,
            chatbot
        ]
    )

    clear_documents_event.then(
        fn=lambda: (
            None,
            "",
            STATUS_READY
        ),
        outputs=[
            pdf_preview,
            question_input,
            assistant_status
        ]
    )


demo.queue()

demo.launch(
    share=True,
    debug=False,
    theme=gr.themes.Base(
        primary_hue="teal",
        neutral_hue="slate"
    ),
    css=CUSTOM_CSS,
    js=APP_JAVASCRIPT,
    footer_links=[]
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b94aa7bc54a74c8ed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
